# Prediction Geometry

Analyzes the geometric structure of how predictions form and sharpen across layers — the shape of the unembedding manifold, calibration of intermediate predictions, and the trajectory through vocabulary space.

**References:**
- Dar et al. (2023) "Analyzing Transformers in Embedding Space"
- Geva et al. (2022) "Transformer Feed-Forward Layers Build Predictions by Promoting Concepts in the Vocabulary Space"

## Why This Matters

Prediction geometry tracks the trajectory of the residual stream through vocabulary space as it approaches the final prediction. Sharpening rate, unembedding alignment, and prediction decomposition reveal the geometric process by which diffuse representations converge to sharp predictions.

**Key references:**
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.prediction_geometry import (
    vocab_projection_trajectory,
    prediction_sharpening_rate,
    unembedding_alignment_per_head,
    token_promotion_geometry,
    final_layer_residual_decomposition,
)

# Create a small model
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35])
print(f"Model: {cfg.n_layers} layers, d_model={cfg.d_model}")

## 1. Vocab Projection Trajectory

At each layer, project through the unembedding to see the top-k predicted tokens. Track when the final prediction enters the top-k and becomes top-1.

In [ ]:
traj = vocab_projection_trajectory(model, tokens, pos=-1, top_k=5)

print(f"Final prediction: token {traj['final_prediction']}")
print(f"First enters top-5 at layer: {traj['final_enters_top_k_layer']}")
print(f"First becomes top-1 at layer: {traj['final_becomes_top_1_layer']}")
print(f"\nLayer-by-layer top tokens:")
for l in range(cfg.n_layers):
    print(f"  Layer {l}: {traj['layer_top_tokens'][l]}")

## 2. Prediction Sharpening Rate

How quickly does the prediction distribution sharpen (entropy decrease) at each layer? The "crystallization layer" is where sharpening is fastest.

In [ ]:
sharp = prediction_sharpening_rate(model, tokens)

for l in range(cfg.n_layers):
    print(f"Layer {l}: entropy={sharp['layer_entropies'][l]:.3f}, "
          f"top-1 prob={sharp['layer_top1_probs'][l]:.4f}")

print(f"\nSharpening rates: {[f'{r:.3f}' for r in sharp['sharpening_rates']]}")
print(f"Crystallization layer: {sharp['crystallization_layer']}")
print(f"Total sharpening: {sharp['total_sharpening']:.3f} nats")

## 3. Unembedding Alignment Per Head

For each attention head, project its output through the unembedding to see which vocabulary items it promotes/demotes. Shows what each head "votes for."

In [ ]:
align = unembedding_alignment_per_head(model, tokens, layer=1, top_k=3)

for h in range(cfg.n_heads):
    print(f"Head {h}:")
    print(f"  Promotes: {list(zip(align['promoted_tokens'][h], "
          f"[f'{v:.3f}' for v in align['promoted_logits'][h]]))}")
    print(f"  Demotes:  {list(zip(align['demoted_tokens'][h], "
          f"[f'{v:.3f}' for v in align['demoted_logits'][h]]))}")
    print(f"  Logit norm: {align['head_logit_norms'][h]:.4f}")

## 4. Token Promotion Geometry

Analyze the geometric relationships between unembedding directions for a set of tokens. Reveals whether semantically similar tokens have nearby directions.

In [ ]:
geom = token_promotion_geometry(model, [0, 1, 2, 3, 4])

print(f"Pairwise similarity matrix:")
for i in range(5):
    row = [f"{geom['pairwise_similarity'][i, j]:.3f}" for j in range(5)]
    print(f"  Token {i}: {row}")

print(f"\nMean pairwise similarity: {geom['mean_pairwise_similarity']:.4f}")
print(f"Mean cosine to centroid: {geom['mean_cosine_to_centroid']:.4f}")
print(f"Norms: {[f'{n:.3f}' for n in geom['norms']]}")

## 5. Final Layer Residual Decomposition

Decompose the final residual stream into per-layer contributions that are parallel vs orthogonal to the prediction direction. Shows which layers build toward the prediction vs do other work.

In [ ]:
decomp = final_layer_residual_decomposition(model, tokens)

print(f"Prediction token: {decomp['prediction_token']}")
print(f"\nParallel contributions (toward prediction):")
labels = ['Embedding'] + [f'Layer {l}' for l in range(cfg.n_layers)]
for i, (label, par, orth) in enumerate(zip(
    labels, decomp['parallel_contributions'], decomp['orthogonal_norms']
)):
    print(f"  {label}: parallel={par:.4f}, orthogonal_norm={orth:.4f}")

print(f"\nTotal parallel: {decomp['total_parallel']:.4f}")
print(f"Dominant component: {labels[decomp['dominant_layer']]}")